In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
analysis = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv')
scoring = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv')

In [3]:
print(f"Loaded Analysis Data: {analysis.shape}")
print(f"Loaded Scoring Data: {scoring.shape}")

Loaded Analysis Data: (40000, 23)
Loaded Scoring Data: (10000, 22)


In [5]:
# Store Customer IDs for submission
scoring_ids = scoring['customer_id']

# ---------------------------------------------------------
# 2. Separate Target & Features
# ---------------------------------------------------------
# For Training Data: Separate Target (y) and Features (X)
target_col = 'monthly_spend'
X = analysis.drop(columns=[target_col, 'customer_id'])
y = analysis[target_col]

# For Scoring Data: Just remove ID (Target doesn't exist here)
X_score = scoring.drop(columns=['customer_id'])

In [6]:
# ---------------------------------------------------------
# 3. Data Cleaning (Imputation)
# ---------------------------------------------------------
# Identify column types
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Define Imputers
imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')

# Fit on TRAINING data, transform BOTH
# (This prevents data leakage from scoring to training)
X[num_cols] = imputer_num.fit_transform(X[num_cols])
X_score[num_cols] = imputer_num.transform(X_score[num_cols])

X[cat_cols] = imputer_cat.fit_transform(X[cat_cols])
X_score[cat_cols] = imputer_cat.transform(X_score[cat_cols])

In [7]:
# ---------------------------------------------------------
# 4. Feature Engineering
# ---------------------------------------------------------
def create_features(df):
    df = df.copy()
    # Interactions
    df['age_annual_income'] = df['age'] * df['annual_income']
    df['credit_score_transactions'] = df['credit_score'] * df['num_transactions']
    df['age_credit_score'] = df['age'] * df['credit_score']
    df['income_transactions'] = df['annual_income'] * df['num_transactions']
    # Polynomials
    df['age_squared'] = df['age'] ** 2
    df['income_squared'] = df['annual_income'] ** 2
    df['credit_score_squared'] = df['credit_score'] ** 2
    return df

X = create_features(X)
X_score = create_features(X_score)

In [8]:
# ---------------------------------------------------------
# 5. One-Hot Encoding
# ---------------------------------------------------------
# Combine briefly to ensure same columns after get_dummies
train_len = len(X)
combined = pd.concat([X, X_score], axis=0)
combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

# Split back apart
X_encoded = combined_encoded.iloc[:train_len, :]
X_score_encoded = combined_encoded.iloc[train_len:, :]

In [10]:
# ---------------------------------------------------------
# 6. Train/Test Split
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=2101
)

In [11]:
# ---------------------------------------------------------
# 7. Robust Gradient Boosting Model
# ---------------------------------------------------------

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

params = {
    'n_estimators': 1000,       # More trees
    'learning_rate': 0.01,      # Slower learning
    'max_depth': 4,             # Shallow trees
    'min_samples_split': 10,    # Constraints to prevent overfitting
    'min_samples_leaf': 5,
    'subsample': 0.8,           # Stochastic boosting (random 80% data)
    'loss': 'squared_error',
    'random_state': 2101
}

print("Training Robust Gradient Boosting Model...")
gb_robust = GradientBoostingRegressor(**params)
gb_robust.fit(X_train, y_train)

Training Robust Gradient Boosting Model...


,loss,'squared_error'
,learning_rate,0.01
,n_estimators,1000
,subsample,0.8
,criterion,'friedman_mse'
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_depth,4
,min_impurity_decrease,0.0
,init,None


In [12]:
# ---------------------------------------------------------
# 8. Evaluation
# ---------------------------------------------------------
test_pred = gb_robust.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("-" * 30)
print(f"✅ Local Test RMSE: {rmse:.4f}")
print("-" * 30)

------------------------------
✅ Local Test RMSE: 256.5590
------------------------------


In [13]:
%%time

# 1. TRANSFORM TARGET (The Main Change)
# We convert 'monthly_spend' to 'log(monthly_spend)'
y_log = np.log1p(y)

# 2. Split Data (Using the new log-target)
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_encoded, y_log, test_size=0.3, random_state=2101
)

# 3. Define Model (Small Change: Increased Estimators)
params = {
    'n_estimators': 1500,       # Increased from 1000 to 1500
    'learning_rate': 0.01,
    'max_depth': 4,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'subsample': 0.8,
    'loss': 'squared_error',
    'random_state': 2101
}

print("Training Log-Transform Model...")
gb_log = GradientBoostingRegressor(**params)
gb_log.fit(X_train, y_train_log)

# 4. Evaluate (Inverse Transform needed!)
# Predict (Result is in Log scale)
pred_log = gb_log.predict(X_test)

# Convert back to Dollars ($) using expm1
pred_real = np.expm1(pred_log)
y_test_real = np.expm1(y_test_log)

rmse = np.sqrt(mean_squared_error(y_test_real, pred_real))
print("-" * 30)
print(f"✅ Local Test RMSE (Log Strategy): {rmse:.4f}")
print("-" * 30)

Training Log-Transform Model...
------------------------------
✅ Local Test RMSE (Log Strategy): 256.4398
------------------------------
CPU times: user 2min 31s, sys: 442 ms, total: 2min 31s
Wall time: 2min 32s


In [14]:
# ---------------------------------------------------------
# 9. Generate Submission
# ---------------------------------------------------------
final_scoring_pred = gb_robust.predict(X_score_encoded)

submission = pd.DataFrame({
    'customer_id': scoring_ids,
    'monthly_spend': final_scoring_pred
})

file_name = 'Submission_Robust_GB_2101.csv'
submission.to_csv(file_name, index=False)
print(f"🎉 Submission saved as: {file_name}")

🎉 Submission saved as: Submission_Robust_GB_2101.csv


In [15]:
# 5. Generate Submission
score_pred_log = gb_log.predict(X_score_encoded)
score_pred_real = np.expm1(score_pred_log)  # Convert back to dollars

submission = pd.DataFrame({
    'customer_id': scoring_ids,
    'monthly_spend': score_pred_real
})

file_name = 'Submission_GB_LogTransform.csv'
submission.to_csv(file_name, index=False)
print(f"🎉 Submission saved as: {file_name}")

🎉 Submission saved as: Submission_GB_LogTransform.csv
